查看缺失情况

In [2]:
import pandas as pd

df = pd.read_csv(r'C:\Users\dongzj\Desktop\AD_tabular\ADNIMERGE_selfuse.csv')

cols = df.columns[0:40]
print("all value:",cols)
sub_df = df[cols]

row_missing_counts = sub_df.isna().sum(axis=1)
missing_stat = row_missing_counts.value_counts().sort_index()

print("=== 每行缺失变量数量分布 ===")
for k in missing_stat.index:
    count = missing_stat[k]
    if k == 0:
        print(f"不缺失的样本有 {count} 个")
    else:
        print(f"缺失 {k} 个变量的样本有 {count} 个")

col_missing = sub_df.isna().sum().sort_values(ascending=False)

print("\n=== 各变量缺失个数 ===")
for col, n_miss in col_missing.items():
    print(f"变量 {col} 缺失 {n_miss} 个")


all value: Index(['RID', 'PTID', 'VISCODE', 'DX_bl', 'DX', 'AGE', 'PTGENDER', 'PTEDUCAT',
       'PTETHCAT', 'PTRACCAT', 'PTMARRY', 'APOE4', 'CDRSB', 'ADAS11', 'ADAS13',
       'ADASQ4', 'MMSE', 'RAVLT_immediate', 'RAVLT_learning',
       'RAVLT_forgetting', 'RAVLT_perc_forgetting', 'LDELTOTAL', 'DIGITSCOR',
       'TRABSCOR', 'FAQ', 'MOCA', 'EcogPtMem', 'EcogPtLang', 'EcogPtVisspat',
       'EcogPtPlan', 'EcogPtOrgan', 'EcogPtDivatt', 'EcogPtTotal', 'EcogSPMem',
       'EcogSPLang', 'EcogSPVisspat', 'EcogSPPlan', 'EcogSPOrgan',
       'EcogSPDivatt', 'EcogSPTotal'],
      dtype='object')
=== 每行缺失变量数量分布 ===
缺失 1 个变量的样本有 4051 个
缺失 2 个变量的样本有 1285 个
缺失 3 个变量的样本有 252 个
缺失 4 个变量的样本有 56 个
缺失 5 个变量的样本有 27 个
缺失 6 个变量的样本有 16 个
缺失 7 个变量的样本有 14 个
缺失 8 个变量的样本有 98 个
缺失 9 个变量的样本有 77 个
缺失 10 个变量的样本有 39 个
缺失 11 个变量的样本有 13 个
缺失 12 个变量的样本有 9 个
缺失 13 个变量的样本有 8 个
缺失 14 个变量的样本有 6 个
缺失 15 个变量的样本有 2528 个
缺失 16 个变量的样本有 1186 个
缺失 17 个变量的样本有 106 个
缺失 18 个变量的样本有 30 个
缺失 19 个变量的样本有 25 个
缺失 20 个变量的样本有 14 个
缺失 21 个

处理分类变量

In [3]:
import pandas as pd

df = pd.read_csv(r'C:\Users\dongzj\Desktop\AD_tabular\ADNIMERGE_selfuse.csv', encoding='utf-8')

cols = ['PTGENDER', 'PTETHCAT', 'PTRACCAT', 'PTMARRY']
mappings = {}

for col in cols:
    uniq = df[col].astype(str).unique()
    mapping = {v: i for i, v in enumerate(uniq)}
    df[col] = df[col].astype(str).map(mapping)
    mappings[col] = mapping

df.to_csv('ADNIMERGE_selfuse_encoded.csv', index=False, encoding='utf-8')

for col, mapping in mappings.items():
    print(f'列 {col} 的替换情况：')
    for k, v in mapping.items():
        print(f'  {k} -> {v}')


列 PTGENDER 的替换情况：
  Male -> 0
  Female -> 1
列 PTETHCAT 的替换情况：
  Not Hisp/Latino -> 0
  Hisp/Latino -> 1
  Unknown -> 2
列 PTRACCAT 的替换情况：
  White -> 0
  More than one -> 1
  Black -> 2
  Asian -> 3
  Am Indian/Alaskan -> 4
  Unknown -> 5
  Hawaiian/Other PI -> 6
列 PTMARRY 的替换情况：
  Married -> 0
  Divorced -> 1
  Widowed -> 2
  Never married -> 3
  Unknown -> 4


划分数据集

In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv(r'C:\Users\dongzj\Desktop\AD_tabular\ADNIMERGE_selfuse_encoded.csv')

cols = df.columns[4:40]
missing_counts = df[cols].isna().sum(axis=1)

rng = np.random.RandomState(42)

train_idx = []
test_idx = []

groups = missing_counts.groupby(missing_counts).groups

for k, idxs in groups.items():
    idxs = np.array(list(idxs))
    rng.shuffle(idxs)
    n = len(idxs)
    n_test = int(round(n * 0.1))
    n_train = n - n_test
    train_idx.extend(idxs[:n_train])
    test_idx.extend(idxs[n_train:])

train_df = df.loc[sorted(train_idx)]
test_df = df.loc[sorted(test_idx)]

print(len(train_df), len(test_df))

train_df.to_csv('train.csv', index=False)
test_df.to_csv('test.csv', index=False)


8885 988
